In [17]:
from edgar import Company, set_identity

set_identity("Kedar Haldar kedar.haldar11@gmail.com")

In [18]:
from pathlib import Path
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

root = Path(
    r"C:\Users\kedar\OneDrive\Documents\GitHub\Finance-GraphRag"
)
folder = root / "data" / "finreflectkg" / "data"

files = sorted(folder.glob("*.parquet"))

print(len(files))       # Number of files
print(files[0].name)    # First filename
print(folder.exists()) # Does the folder exist?

103
train-00000-of-00103.parquet
True


In [19]:
df = pd.read_parquet(files[0])
print("File:", files[0].name)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(3))

File: train-00000-of-00103.parquet
Shape: (170033, 18)
Columns: ['triplet_id', 'entity', 'entity_type', 'relationship', 'target', 'target_type', 'start_date', 'end_date', 'extraction_type', 'ticker', 'year', 'source_file', 'page_id', 'chunk_id', 'chunk_text', 'triplet_length', 'chunk_text_length', 'has_context']


,triplet_id,entity,entity_type,relationship,target,target_type,start_date,end_date,extraction_type,ticker,year,source_file,page_id,chunk_id,chunk_text,triplet_length,chunk_text_length,has_context
0,ctas_2014_page_1_chunk_1_triplet_1,ctas,ORG,subject_to,sec regulation,REGULATORY_REQUIREMENT,January 2014,default_end_timestamp,default,CTAS,2014,CTAS_10k_2014.pdf,page_1,chunk_1,## UNITED STATES SECURITIES AND EXCHANGE COMMI...,30,3578,True
1,ctas_2014_page_1_chunk_1_triplet_2,ctas,ORG,operates_in,united state,GPE,January 2014,default_end_timestamp,default,CTAS,2014,CTAS_10k_2014.pdf,page_1,chunk_1,## UNITED STATES SECURITIES AND EXCHANGE COMMI...,29,3578,True
2,ctas_2014_page_1_chunk_1_triplet_3,ctas,ORG,discloses,common stock,FIN_INST,November 2013,default_end_timestamp,default,CTAS,2014,CTAS_10k_2014.pdf,page_1,chunk_1,## UNITED STATES SECURITIES AND EXCHANGE COMMI...,27,3578,True


In [20]:
display(df["relationship"].value_counts().head(30))

relationship
discloses             73824
depends_on             9492
subject_to             9206
operates_in            9033
negatively_impacts     7451
impacted_by            5940
has_stake_in           4985
increase               4180
introduces             3928
face                   3373
decrease               2669
invests_in             2136
announces              2086
produce                1754
complies_with          1740
guides_on              1540
partners_with          1521
positively_impacts     1345
impact                  998
regulates               801
offer                   787
supply                  757
include                 719
involved_in             690
us                      682
issue                   640
competes_with           613
serves_as               504
recognizes              479
provide                 476
Name: count, dtype: int64

In [21]:
sample = df[df["relationship"] == "discloses"].sample(
    n=5, random_state=42
)

for _, row in sample.iterrows():
    print("ID:", row["triplet_id"])
    print("Claim:", row["entity"], "→", row["relationship"], "→", row["target"])
    print("Source:", row["source_file"])
    print("Text:", row["chunk_text"])
    print("-" * 80)

ID: well_2024_page_77_chunk_1_triplet_74
Claim: well → discloses → total current liability
Source: WELL_10k_2024.pdf
Text: Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
--------------------------------------------------------------------------------
ID: cnp_2020_page_3_chunk_2_triplet_6
Claim: cnp → discloses → financial statement and supplementary data
Source: CNP_10k_2020.pdf
Text: | | | Page | |----------|--------------------------------------------------------------------------------------------------------------|----------| | Item 1. | Business | 1 | | Item 1A. | Risk Factors | 20 | | Item 1B. | Unresolved Staff Comments | 40 | | Item 2. | Properties | 41 | | Item 3. | Legal Proceedings | 43 | | Item 4. | Mine Safety Disclosures | 43 | | PART II | PART II | PART II | | Item 5. | Market for Registrants' Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities | 43 | | Item 6. | Selected Financial Data | 43 

In [22]:
amazon = sample[
    sample["triplet_id"] == "amzn_2017_page_56_chunk_5_triplet_9"
]

print(amazon[[
    "entity", "relationship", "target",
    "start_date", "end_date", "year", "source_file"
]].to_string(index=False))

entity relationship                   target    start_date      end_date  year       source_file
  amzn    discloses no borrowing outstanding December 2015 December 2015  2017 AMZN_10k_2017.pdf


### Example: Amazon borrowing claim

Triple ID: amzn_2017_page_56_chunk_5_triplet_9

- Graph claim: Amazon discloses no borrowing outstanding.
- Stored dates: December 2015 to December 2015.
- Source text: No borrowings under a specific credit agreement
  as of December 31, 2015 and 2014.
- Issue: The triple omits the credit-agreement scope and the
  additional 2014 date.
- XBRL verification: Not yet attempted. Requires a fact for
  that facility and date; total company debt is not equivalent.

In [23]:
changes = df[
    df["relationship"].isin(["increase", "decrease"])
].sample(n=5, random_state=42)

for _, row in changes.iterrows():
    print("ID:", row["triplet_id"])
    print("Claim:", row["entity"], "→", row["relationship"], "→", row["target"])
    print("Dates:", row["start_date"], "→", row["end_date"])
    print("Source:", row["source_file"])
    print("Text:", row["chunk_text"])
    print("-" * 80)

ID: cnp_2017_page_48_chunk_1_triplet_20
Claim: cnp → increase → compliance expense
Dates: January 2017 → December 2017
Source: CNP_10k_2017.pdf
Text: temperatures for the rest of the year, including the summer months, in the Houston area. In 2013, we experienced a colder than normal spring and very cold weather in November and December in Houston and all of the states in which we have gas customers. Our long-term national trends indicate customers have reduced their energy consumption, and reduced consumption can adversely affect our results. However, due to more affordable energy prices and continued economic improvement in the areas we serve, the trend toward lower usage has slowed in some of the areas we serve. In addition, in many of our service areas, particularly in the Houston area and in Minnesota, we have benefited from a growth in the number of customers that also tends to mitigate the effects of reduced consumption. We anticipate that this trend will continue as the regions'

In [24]:
candidate = df.loc[
    df["triplet_id"] == "vz_2015_page_28_chunk_2_triplet_4"
].iloc[0]

print(candidate["triplet_id"])
print(candidate["chunk_text"])

vz_2015_page_28_chunk_2_triplet_4
| | | | | | (dollars in millions) | |--------------------------------------------------|--------------------------------|--------------------|--------------------------------------|------------------------|-------------------------| | | | Additions | Additions | | | | Description | Balance at Beginning of Period | Chargedto Expenses | Chargedto Other Accounts Note (a)(b) | Deductions Note (c)(d) | Balance at Endof Period | | Allowance for Uncollectible Accounts Receivable: | | | | | | | Year 2015 (e) | $ 739 | $ 1,610 | $ 200 | $ 1,512 | $ 1,037 | | Year 2014 | 645 | 1,095 | 141 | 1,142 | 739 | | Year 2013 | 641 | 993 | 162 | 1,151 | 645 | | Valuation Allowance for Deferred TaxAssets: | | | | | | | Year 2015 | $ 1,841 | $ 237 | $ 1,701 | $ 365 | $ 3,414 | | Year 2014 | 1,685 | 505 | 5 | 354 | 1,841 | | Year 2013 | 2,096 | 235 | 64 | 710 | 1,685 |


In [25]:
verizon = Company("VZ")
vz_facts = verizon.get_facts()
vz_df = vz_facts.query().to_dataframe()

allowance_concepts = vz_df[
    vz_df["concept"].str.contains(
        "Allowance|Uncollectible|Doubtful",
        case=False,
        na=False
    )
][["concept", "label"]].drop_duplicates()

display(allowance_concepts)
print(allowance_concepts.to_string(index=False))

,concept,label
1122,us-gaap:AllowanceForDoubtfulAccountsPremiumsAn...,"Allowance for Doubtful Accounts, Premiums and ..."
1130,us-gaap:AllowanceForDoubtfulAccountsReceivable...,"Accounts Receivable, Allowance for Credit Loss..."
6807,us-gaap:DeferredTaxAssetsTaxDeferredExpenseRes...,"Deferred Tax Assets, Tax Deferred Expense, Res..."
6815,us-gaap:DeferredTaxAssetsValuationAllowance,"Deferred Tax Assets, Valuation Allowance"
10018,us-gaap:FinancingReceivableAllowanceForCreditL...,"Financing Receivable, Allowance for Credit Loss"
10139,us-gaap:FinancingReceivableAllowanceForCreditL...,"Financing Receivable, Allowance for Credit Los..."
24123,us-gaap:ProvisionForDoubtfulAccounts,"Accounts Receivable, Credit Loss Expense (Reve..."
27530,us-gaap:ValuationAllowancesAndReservesPeriodIn...,"SEC Schedule, 12-09, Valuation Allowances and ..."
28785,us-gaap:EffectiveIncomeTaxRateReconciliationCh...,"Effective Income Tax Rate Reconciliation, Chan..."
28827,us-gaap:IncomeTaxReconciliationChangeInDeferre...,"Effective Income Tax Rate Reconciliation, Chan..."


                                                                                   concept                                                                                                label
                           us-gaap:AllowanceForDoubtfulAccountsPremiumsAndOtherReceivables                                      Allowance for Doubtful Accounts, Premiums and Other Receivables
                                     us-gaap:AllowanceForDoubtfulAccountsReceivableCurrent                                              Accounts Receivable, Allowance for Credit Loss, Current
us-gaap:DeferredTaxAssetsTaxDeferredExpenseReservesAndAccrualsAllowanceForDoubtfulAccounts    Deferred Tax Assets, Tax Deferred Expense, Reserves and Accruals, Allowance for Doubtful Accounts
                                               us-gaap:DeferredTaxAssetsValuationAllowance                                                             Deferred Tax Assets, Valuation Allowance
                                       u

In [26]:
concept = "us-gaap:AllowanceForDoubtfulAccountsReceivableCurrent"

matches = vz_df[
    (vz_df["concept"] == concept)
    & pd.to_datetime(vz_df["period_end"]).isin(
        pd.to_datetime(["2013-12-31", "2014-12-31"])
    )
    & (vz_df["form_type"] == "10-K")
]

print(matches[[
    "period_end",
    "numeric_value",
    "unit",
    "filing_date",
    "accession"
]].sort_values(["filing_date", "period_end"]).to_string(index=False))

period_end  numeric_value unit filing_date            accession
2013-12-31    645000000.0  USD  2014-02-27 0001193125-14-073266
2013-12-31    645000000.0  USD  2015-02-23 0001193125-15-057710
2014-12-31    739000000.0  USD  2015-02-23 0001193125-15-057710
2014-12-31    739000000.0  USD  2016-02-23 0001193125-16-473367


In [ ]:
filing_id = "0001193125-15-057710"

pair = matches[
    (matches["accession"] == filing_id)
    & (matches["unit"] == "USD")
].copy()

pair["period_end"] = pd.to_datetime(pair["period_end"])
assert len(pair) == 2 and pair["period_end"].nunique() == 2

values = pair.set_index("period_end")["numeric_value"]
before = values.loc[pd.Timestamp("2013-12-31")]
after = values.loc[pd.Timestamp("2014-12-31")]

print("Change in USD:", after - before)
print("Supports 'increase':", after > before)

Change in USD: 94000000.0
Supports 'increase': False


### Mini test 1: Verizon allowance increased

- Triple ID: vz_2015_page_28_chunk_2_triplet_4
- Graph claim: Verizon's allowance for uncollectible accounts
  receivable increased during 2014.
- XBRL concept: us-gaap:AllowanceForDoubtfulAccountsReceivableCurrent
- Accession: 0001193125-15-057710
- Unit: USD
- 2013-12-31: 645,000,000
- 2014-12-31: 739,000,000
- Change: +94,000,000
- Result: Direction supported, conditional on the concept mapping.
- Remaining checks: Confirm that the current-receivables concept
  matches the source table's scope and resolve the source-filing identity.